In [1]:
from collections import Counter
import pandas as pd

In [3]:
file_path = "Processed_Reviews.csv"
df = pd.read_csv(file_path)

In [4]:
tokenized_reviews = df['tokenized'].dropna().apply(eval)

In [5]:
print(tokenized_reviews.head())

0    [product, arrive, time, packaging, great, qual...
1                               [product, amaze, love]
2            [buy, phone, hz, display, totally, worth]
3              [wow, product, awesome, bit, expensive]
4                      [laptop, work, perfectly, fine]
Name: tokenized, dtype: object


In [6]:
all_words = [word for review in tokenized_reviews for word in review]
unique_words = list(set(all_words))

In [7]:
print(len(unique_words))

53


In [8]:
word_freq = Counter(all_words)
sorted_word_freq = dict(sorted(word_freq.items(), key=lambda item: item[1], reverse=True))

In [9]:
print(list(sorted_word_freq.items())[:10])

[('product', 7), ('quality', 3), ('great', 2), ('amazing', 2), ('love', 2), ('awesome', 2), ('work', 2), ('perfectly', 2), ('life', 2), ('expect', 2)]


In [10]:
document_vectors = []

for review in tokenized_reviews:
    document_vector = [1 if word in review else 0 for word in sorted_word_freq.keys()]
    document_vectors.append(document_vector)

In [11]:
doc_vectors_df = pd.DataFrame(document_vectors, columns=sorted_word_freq.keys())

In [12]:
doc_vectors_df.to_csv("document_vectors.csv", index=False)

In [13]:
word_freq_df = pd.DataFrame(list(sorted_word_freq.items()), columns=["Word", "Frequency"])

print("Word Frequency Table:")
print(word_freq_df.head())

Word Frequency Table:
      Word  Frequency
0  product          7
1  quality          3
2    great          2
3  amazing          2
4     love          2


In [14]:
import math
from collections import Counter

In [15]:
documents = tokenized_reviews.tolist()

In [16]:
print(documents[0])

['product', 'arrive', 'time', 'packaging', 'great', 'quality', 'amazing']


In [17]:
def compute_tf(document):
    word_count = Counter(document)
    tf = {word: count / len(document) for word, count in word_count.items()}
    return tf

In [18]:
def compute_idf(documents):
    N = len(documents)
    idf = {}

    all_words = set(word for doc in documents for word in doc)

    for word in all_words:
        count = sum(1 for doc in documents if word in doc)
        idf[word] = math.log(N / count)

    return idf

In [19]:
idf = compute_idf(documents)

In [20]:
print(list(idf.items())[:5])

[('cable', 2.5649493574615367), ('totally', 2.5649493574615367), ('battery', 2.5649493574615367), ('delivery', 2.5649493574615367), ('not', 2.5649493574615367)]


In [21]:
tf_data = [compute_tf(doc) for doc in documents]

In [22]:
tf_df = pd.DataFrame(tf_data).fillna(0)

In [23]:
tf_df.to_csv("tf_scores.csv", index=False)

In [24]:
idf_df = pd.DataFrame([idf]).fillna(0)
idf_df.to_csv("idf_scores.csv", index=False)

In [25]:
def compute_tfidf(document, idf):
    tfidf = {}
    tf = compute_tf(document)

    for word, tf_value in tf.items():
        tfidf[word] = tf_value * idf[word]

    return tfidf

In [26]:
tfidf_data = [compute_tfidf(doc, idf) for doc in documents]

tfidf_df = pd.DataFrame(tfidf_data).fillna(0)

tfidf_df.to_csv("tfidf_scores.csv", index=False)